In [15]:
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.preprocessing import MinMaxScaler

In [16]:
df = pd.read_csv("../data/processed/marketing_campaign_cleaned.csv")

df.head()

,ID,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,Complain,Response,Age,Children,Total_Spending,Customer_Tenure,Total_Purchases,Average_Spend_Per_Purchase,Cluster,Customer_Segment
0,5524,1957,Graduation,Single,58138.0,0,0,2012-09-04,58,635,...,0,1,69,0,1617,5071,22,70.304348,3,Premium VIP Customers
1,2174,1954,Graduation,Single,46344.0,1,1,2014-03-08,38,11,...,0,0,72,2,27,4521,4,5.400000,0,Low-Value Customers
2,4141,1965,Graduation,Together,71613.0,0,0,2013-08-21,26,426,...,0,0,61,0,776,4720,20,36.952381,1,Loyal Customers
3,6182,1984,Graduation,Together,26646.0,1,0,2014-02-10,26,11,...,0,0,42,1,53,4547,6,7.571429,0,Low-Value Customers
4,5324,1981,PhD,Married,58293.0,1,0,2014-01-19,94,173,...,0,0,45,1,422,4569,14,28.133333,2,Dormant Customers


In [17]:
clv_model = joblib.load("../models/clv_model.pkl")

campaign_model = joblib.load("../models/campaign_response_model.pkl")

In [18]:
clv_features = [

    "Income",
    "Age",
    "Recency",
    "NumWebPurchases",
    "NumStorePurchases",
    "NumCatalogPurchases",
    "NumDealsPurchases",
    "NumWebVisitsMonth",
    "Kidhome",
    "Teenhome"

]

X_clv = df[clv_features]

In [23]:
X_clv = df[clv_features]

df["Customer_Value_Score"] = clv_model.predict(X_clv)
df[["Customer_Value_Score"]].head()

,Customer_Value_Score
0,12555.233367
1,91.134267
2,11418.041767
3,257.733867
4,533.556033


In [21]:
campaign_features = [

    "Income",
    "Age",
    "Recency",

    "Kidhome",
    "Teenhome",

    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases",

    "NumWebVisitsMonth",
    "NumDealsPurchases",

    "Total_Spending",
    "Total_Purchases"

]

X_campaign = df[campaign_features]

In [22]:
X_campaign = df[campaign_features]

df["Response_Probability"] = campaign_model.predict_proba(
    X_campaign
)[:,1]

In [26]:
scaler = MinMaxScaler()

df["CVS_Normalized"] = scaler.fit_transform(
    df[["Customer_Value_Score"]]
)

df["Income_Normalized"] = scaler.fit_transform(
    df[["Income"]]
)

df["Spending_Normalized"] = scaler.fit_transform(
    df[["Total_Spending"]]
)

df["Recency_Normalized"] = 1 - scaler.fit_transform(
    df[["Recency"]]
)

In [27]:
segment_value = (
    df.groupby("Cluster")["Total_Spending"]
      .mean()
      .sort_values()
)

segment_rank = (
    segment_value.rank(method="dense")
    / segment_value.rank(method="dense").max()
)

df["Segment_Score"] = df["Cluster"].map(segment_rank)

In [28]:
df["Opportunity_Score"] = (

    0.30 * df["CVS_Normalized"] +

    0.30 * df["Response_Probability"] +

    0.15 * df["Spending_Normalized"] +

    0.10 * df["Income_Normalized"] +

    0.10 * df["Segment_Score"] +

    0.05 * df["Recency_Normalized"]

) * 100

In [29]:
def recommend(row):

    score = row["Opportunity_Score"]
    response = row["Response_Probability"]

    if score >= 80 and response >= 0.70:
        return "Premium Loyalty Campaign"

    elif score >= 80:
        return "Re-engagement Campaign"

    elif score >= 60 and response >= 0.50:
        return "Cross-Sell Campaign"

    elif score >= 40:
        return "Email Nurture Campaign"

    else:
        return "Low Priority"

df["Recommendation"] = df.apply(recommend, axis=1)

In [31]:
df["Expected_Revenue"] = (
    df["Response_Probability"] *
    df["Customer_Value_Score"]
)

df["Business_Impact"] = pd.cut(
    df["Expected_Revenue"],
    bins=[0,5000,15000,np.inf],
    labels=["Low","Medium","High"]
)

In [33]:
def priority(score):

    if score>=85:
        return "Critical"

    elif score>=70:
        return "High"

    elif score>=55:
        return "Medium"

    else:
        return "Low"

df["Priority"]=df[
    "Opportunity_Score"
].apply(priority)

In [34]:
final_output = df[
    [
        "ID",
        "Cluster",
        "Income",
        "Total_Spending",
        "Customer_Value_Score",
        "Response_Probability",
        "Opportunity_Score",
        "Recommendation",
        "Priority",
        "Expected_Revenue",
        "Business_Impact"
    ]
]

final_output.head()

,ID,Cluster,Income,Total_Spending,Customer_Value_Score,Response_Probability,Opportunity_Score,Recommendation,Priority,Expected_Revenue,Business_Impact
0,5524,3,58138.0,1617,12555.233367,0.786332,54.975464,Email Nurture Campaign,Low,9872.579577,Medium
1,2174,0,46344.0,27,91.134267,0.148889,10.911803,Low Priority,Low,13.568855,Low
2,4141,1,71613.0,776,11418.041767,0.276894,33.201485,Low Priority,Low,3161.582886,Low
3,6182,0,26646.0,53,257.733867,0.067808,9.061723,Low Priority,Low,17.476535,Low
4,5324,2,58293.0,422,533.556033,0.143420,13.263034,Low Priority,Low,76.522714,Low


In [35]:
os.makedirs("../results", exist_ok=True)

df.to_csv(
    "../results/marketsphere_predictions.csv",
    index=False
)

print("✅ Decision Impact Engine Completed")

✅ Decision Impact Engine Completed
